# COVID-19 Mortality Comparative Analysis

Portfolio-grade notebook for a data analyst job application. This notebook demonstrates data cleaning, KPI engineering, exploratory analysis, wave detection, and country segmentation.

In [ ]:

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from scipy.signal import find_peaks
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans

DATA_PATH = Path('../data/raw/daily-new-confirmed-covid-19-deaths-per-million-people.csv')
df = pd.read_csv(DATA_PATH, parse_dates=['Day']).rename(columns={'New deaths (per 1M)': 'deaths_per_million'})
df = df.sort_values(['Entity', 'Day']).reset_index(drop=True)
df['rolling_7d'] = df.groupby('Entity')['deaths_per_million'].transform(lambda s: s.rolling(7, min_periods=1).mean())
df['year'] = df['Day'].dt.year
df['month'] = df['Day'].dt.to_period('M').astype(str)
df.head()


## 1. Data quality check
We first verify shape, null values, date range, and country coverage.

In [ ]:

print('Shape:', df.shape)
print('Countries:', df['Entity'].nunique())
print('Date range:', df['Day'].min().date(), 'to', df['Day'].max().date())
display(df.isna().sum().to_frame('null_count'))
display(df.groupby('Entity').size().to_frame('rows_per_country'))


## 2. Country-level KPIs
These KPIs are designed like a stakeholder scorecard rather than a simple descriptive table.

In [ ]:

def summarize_country(g):
    g = g.sort_values('Day').reset_index(drop=True)
    peak_idx = g['rolling_7d'].idxmax()
    peak_value = float(g.loc[peak_idx, 'rolling_7d'])
    peak_date = pd.Timestamp(g.loc[peak_idx, 'Day'])
    peaks, _ = find_peaks(g['rolling_7d'].values, prominence=max(peak_value * 0.08, 0.15), distance=45)
    post = g[g['Day'] >= peak_date]
    recovery_match = post[post['rolling_7d'] <= peak_value * 0.25]
    recovery_days = int((recovery_match.iloc[0]['Day'] - peak_date).days) if not recovery_match.empty else np.nan
    return pd.Series({
        'peak_7d_deaths_per_million': peak_value,
        'peak_date': peak_date.date().isoformat(),
        'cumulative_deaths_per_million': float(g['deaths_per_million'].sum()),
        'avg_daily_deaths_per_million': float(g['deaths_per_million'].mean()),
        'volatility_std': float(g['deaths_per_million'].std()),
        'estimated_wave_count': int(len(peaks)),
        'days_to_recover_from_main_peak': recovery_days,
        'zero_share': float((g['deaths_per_million'] == 0).mean())
    })

summary = df.groupby('Entity', group_keys=False).apply(summarize_country).reset_index()
summary.sort_values('cumulative_deaths_per_million', ascending=False)


## 3. Trend analysis
The 7-day rolling average removes daily reporting noise and highlights wave structure.

In [ ]:

plt.figure(figsize=(12, 6))
for entity, g in df.groupby('Entity'):
    plt.plot(g['Day'], g['rolling_7d'], label=entity)
plt.title('7-Day Rolling COVID-19 Deaths per Million by Country')
plt.xlabel('Date')
plt.ylabel('Deaths per million')
plt.legend(frameon=False, ncol=2)
plt.tight_layout()
plt.show()


## 4. Annual burden comparison
This view makes it easy to compare how the pandemic burden shifted over time.

In [ ]:

annual = df.groupby(['Entity', 'year'], as_index=False)['deaths_per_million'].sum()
annual_pivot = annual.pivot(index='Entity', columns='year', values='deaths_per_million')
display(annual_pivot.round(1))

fig, ax = plt.subplots(figsize=(8, 4.8))
im = ax.imshow(annual_pivot.values, aspect='auto')
ax.set_xticks(range(len(annual_pivot.columns)), labels=annual_pivot.columns)
ax.set_yticks(range(len(annual_pivot.index)), labels=annual_pivot.index)
ax.set_title('Annual Deaths per Million Heatmap')
for i in range(annual_pivot.shape[0]):
    for j in range(annual_pivot.shape[1]):
        ax.text(j, i, f"{annual_pivot.iloc[i, j]:.0f}", ha='center', va='center', fontsize=8)
fig.colorbar(im, ax=ax, fraction=0.03, pad=0.04)
plt.tight_layout()
plt.show()


## 5. Wave detection and recovery analysis
We use signal processing logic to estimate wave count and evaluate how long each country needed to recover after its main peak.

In [ ]:

summary[['Entity', 'estimated_wave_count', 'days_to_recover_from_main_peak']].sort_values(
    'days_to_recover_from_main_peak', ascending=False
)


## 6. Country segmentation with K-Means
Clustering helps turn descriptive metrics into a comparative framework that can support dashboard filtering, market-style segmentation, or monitoring tiers.

In [ ]:

feature_cols = [
    'peak_7d_deaths_per_million',
    'cumulative_deaths_per_million',
    'volatility_std',
    'estimated_wave_count',
    'days_to_recover_from_main_peak',
    'zero_share'
]
X = StandardScaler().fit_transform(summary[feature_cols])
km = KMeans(n_clusters=3, n_init=20, random_state=42)
summary['cluster'] = km.fit_predict(X)
summary[['Entity', 'cluster'] + feature_cols].sort_values('cluster')


In [ ]:

plt.figure(figsize=(8, 6))
for cluster, g in summary.groupby('cluster'):
    plt.scatter(g['peak_7d_deaths_per_million'], g['cumulative_deaths_per_million'], s=140, label=f'Cluster {cluster}')
    for _, row in g.iterrows():
        plt.annotate(row['Entity'], (row['peak_7d_deaths_per_million'], row['cumulative_deaths_per_million']), xytext=(4,4), textcoords='offset points', fontsize=8)
plt.xlabel('Peak 7-day deaths per million')
plt.ylabel('Cumulative deaths per million')
plt.title('Peak Severity vs Cumulative Burden')
plt.legend(frameon=False)
plt.tight_layout()
plt.show()


## 7. Final takeaways
- The United Kingdom stands out as an extreme-peak outlier.
- The United States and France show very high cumulative burden, even without matching the UK peak.
- India and Canada form a lower-burden group in this sample.
- Recovery speed adds useful context that pure totals would miss.

This kind of framing is much stronger in an interview than just saying which country had the highest value.